# Data Curation

In [ ]:
import json
import sys
import pandas as pd
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.constants import LOCAL_DATA_DIR, LANGUAGES, LOCAL_RESULTS_DIR, DATASPLITS_GMMLU

## MKQA

### Inspect

In [ ]:
import json
from collections import Counter
f_input = LOCAL_DATA_DIR + "/mkqa/mkqa.jsonl"

In [ ]:
## Find the number of languages present in the dataset
languages = set()
with open(f_input, "r", encoding="utf-8") as fin:
    for line in fin:
        example = json.loads(line)
        languages.update(example.get("answers", {}).keys())

print(f"Number of languages: {len(languages)}")
print(f"Languages: {sorted(languages)}")

In [ ]:
## Find all possible answer types
types_set = set()
with open(f_input, "r", encoding="utf-8") as fin:
    for line in fin:
        example = json.loads(line)
        answers = example.get("answers", {})
        for lang_answers in answers.values():
            for ans in lang_answers:
                answer_type = ans.get("type")
                if answer_type:
                    types_set.add(answer_type)
print("Unique types in MKQA:")
for t in sorted(types_set):
    print(f"- {t}")

In [ ]:
## Count the number of questions per answer type
type_counts = Counter()

with open(f_input, "r", encoding="utf-8") as fin:
    for line in fin:
        example = json.loads(line)
        answers = example.get("answers", {})
        
        # collect all distinct types for this question
        this_q_types = set()
        for lang_answers in answers.values():
            for ans in lang_answers:
                t = ans.get("type")
                if t:
                    this_q_types.add(t)
        
        tup = tuple(this_q_types)
        type_counts[tup] += 1


# print out results
print("Question counts by answer type:")
for t, cnt in type_counts.most_common():
    print(f"- {t}: {cnt}")

In [ ]:
## Print some examples
type_counts = Counter()
# to store (query, en_text) for questions with "entity"
entity_examples = []
with open(LOCAL_DATA_DIR, "r", encoding="utf-8") as fin:
    for line in fin:
        example = json.loads(line)
        query = example.get("query")
        answers = example.get("answers", {})
        
        # collect all distinct types for this question
        this_q_types = set()
        for lang_answers in answers.values():
            for ans in lang_answers:
                t = ans.get("type")
                if t:
                    this_q_types.add(t)
        
        # increment count for each type seen in this question
        for t in this_q_types:
            type_counts[t] += 1
        
        # if this question has an "entity" answer, capture the EN text
        if "entity" in this_q_types and len(entity_examples) < 10:
            # assume at least one English answer exists
            en_texts = [ans.get("text") for ans in answers.get("en", []) if ans.get("text")]
            # you can choose first, or join multiple
            first_text = en_texts[0] if en_texts else "(no en text)"
            entity_examples.append((query, first_text))

# print a few "entity" examples with their English answer
print("\nSample 'entity' queries and their English answer text:")
for q, txt in entity_examples:
    print(f"- Q: {q!r}\n  A (en): {txt!r}\n")


### Separate the questions based on type, exclude duplicates, and check remaining questions after time filtering

In [ ]:
import pandas as pd
import json
import os
MKQA_PATH = LOCAL_DATA_DIR + "/mkqa/"

In [ ]:
def load_example_ids(jsonl_path):
    ids = set()
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if 'example_id' in obj:
                    ids.add(obj['example_id'])
            except json.JSONDecodeError:
                continue
    return ids

In [ ]:
## Keep only the answerable queries
output_path = "mkqa_answerable.jsonl"
with open(MKQA_PATH, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
    for line in fin:
        example = json.loads(line)
        answers = example.get("answers", {})
        # check if any language has an unanswerable type
        has_unanswerable = any(any(ans.get("type") == "unanswerable" for ans in lang_answers) for lang_answers in answers.values())
        # write only if the example is NOT unanswerable
        if not has_unanswerable:
            fout.write(json.dumps(example, ensure_ascii=False) + "\n")

In [ ]:
## Keep the binary-type questions from answerable
input_path = MKQA_PATH + "mkqa_answerable.jsonl"
output_path = MKQA_PATH + "mkqa_answerable_binary.jsonl"
with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
    for line in fin:
        example = json.loads(line)
        answers = example.get("answers", {})
        # check if any answer has type "binary"
        has_binary = any(any(ans.get("type") == "binary" for ans in lang_answers) for lang_answers in answers.values())
        if has_binary:
            fout.write(json.dumps(example, ensure_ascii=False) + "\n")

In [ ]:
## Keep the entity-type questions from answerable
input_path = MKQA_PATH + "mkqa_answerable.jsonl"
output_path = MKQA_PATH + "mkqa_answerable_entity.jsonl"
with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
    for line in fin:
        example = json.loads(line)
        answers = example.get("answers", {})
        # check if any answer has type "entity"
        has_entity = any(any(ans.get("type") == "entity" for ans in lang_answers) for lang_answers in answers.values())
        if has_entity:
            fout.write(json.dumps(example, ensure_ascii=False) + "\n")

In [ ]:
## Keep the short_phrase-type questions from answerable
input_path = MKQA_PATH + "mkqa_answerable.jsonl"
output_path = MKQA_PATH + "mkqa_answerable_short_phrase.jsonl"
with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
    for line in fin:
        example = json.loads(line)
        answers = example.get("answers", {})
        # check if any answer has type "short_phrase"
        has_short_phrase = any(any(ans.get("type") == "short_phrase" for ans in lang_answers) for lang_answers in answers.values())
        if has_short_phrase:
            fout.write(json.dumps(example, ensure_ascii=False) + "\n")

In [ ]:
no_duplicates_entity_file  = LOCAL_DATA_DIR + "mkqa_answerable_entity_no_duplicates.jsonl"

# load the IDs from the short‑phrase file
entity_file = MKQA_PATH + "mkqa_answerable_entity.jsonl"
short_phrase_file = MKQA_PATH + "mkqa_answerable_short_phrase.jsonl"
short_ids = load_example_ids(short_phrase_file)

# write only those entity records whose example_id is NOT in short_ids
with open(entity_file, 'r', encoding='utf-8') as fin, \
     open(no_duplicates_entity_file, 'w', encoding='utf-8') as fout:
    for line in fin:
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        eid = obj.get('example_id')
        if eid not in short_ids:
            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

print(f"wrote filtered entities to {no_duplicates_entity_file}")

In [ ]:
## Keep the date-type questions from answerable
## that do not have type entity or short-phrase to avoid duplicates
input_path = MKQA_PATH + "mkqa_answerable.jsonl"
output_path = MKQA_PATH + "mkqa_answerable_date_no_duplicates.jsonl"
disallowed = {"short_phrase", "entity"}

count_total = 0
count_kept = 0

with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
    for line in fin:
        count_total += 1
        example = json.loads(line)
        answers = example.get("answers", {})

        # gather all types present across all languages for this example
        types = set()
        for lang_answers in answers.values():
            if isinstance(lang_answers, list):
                for ans in lang_answers:
                    t = ans.get("type")
                    if isinstance(t, str):
                        types.add(t.strip().lower())

        # Keep if it has 'date' but none of the disallowed types
        if "date" in types and not (types & disallowed):
            fout.write(json.dumps(example, ensure_ascii=False) + "\n")
            count_kept += 1

print(f"Total examples processed: {count_total}")
print(f"Examples kept (date only, no entity/short_phrase): {count_kept}")

In [ ]:
## Keep the number-type questions from answerable
## that do not have type date, entity or short-phrase to avoid duplicates
input_path = MKQA_PATH + "mkqa_answerable.jsonl"
output_path = MKQA_PATH + "mkqa_answerable_number_no_duplicates.jsonl"
disallowed = {"short_phrase", "entity", "date"}

count_total = 0
count_kept = 0

with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
    for line in fin:
        count_total += 1
        example = json.loads(line)
        answers = example.get("answers", {})

        # gather all types present across all languages for this example
        types = set()
        for lang_answers in answers.values():
            if isinstance(lang_answers, list):
                for ans in lang_answers:
                    t = ans.get("type")
                    if isinstance(t, str):
                        types.add(t.strip().lower())

        # Keep if it has 'number' but none of the disallowed types
        if "number" in types and not (types & disallowed):
            fout.write(json.dumps(example, ensure_ascii=False) + "\n")
            count_kept += 1

print(f"Total examples processed: {count_total}")
print(f"Examples kept (number only, no short_phrase/entity/date): {count_kept}")

In [ ]:
## Keep the number with unit-type questions from answerable
## that do not have type date, entity or short-phrase to avoid duplicates
input_path = MKQA_PATH + "mkqa_answerable.jsonl"
output_path = MKQA_PATH + "mkqa_answerable_number_with_unit_no_duplicates.jsonl"
disallowed = {"short_phrase", "entity", "date", "number"}

count_total = 0
count_kept = 0

with open(input_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
    for line in fin:
        count_total += 1
        example = json.loads(line)
        answers = example.get("answers", {})

        # gather all types present across all languages for this example
        types = set()
        for lang_answers in answers.values():
            if isinstance(lang_answers, list):
                for ans in lang_answers:
                    t = ans.get("type")
                    if isinstance(t, str):
                        types.add(t.strip().lower())

        # Keep if it has 'number_with_unit' but none of the disallowed types
        if "number_with_unit" in types and not (types & disallowed):
            fout.write(json.dumps(example, ensure_ascii=False) + "\n")
            count_kept += 1

print(f"Total examples processed: {count_total}")
print(f"Examples kept (number only, no short_phrase/entity/date/number): {count_kept}")

In [ ]:
## Sanity check: check file overlap 
short_phrase_file = MKQA_PATH + "mkqa_answerable_short_phrase.jsonl"
entity_file = MKQA_PATH + "mkqa_answerable_entity_no_duplicates.jsonl"
short_phrase_file = MKQA_PATH + "mkqa_answerable_short_phrase.jsonl"
date_file = MKQA_PATH + "mkqa_answerable_date_no_duplicates.jsonl"
number_file = MKQA_PATH + "mkqa_answerable_number_no_duplicates.jsonl"
number_with_unit_file = MKQA_PATH + "mkqa_answerable_number_with_unit_no_duplicates.jsonl"

# load all ID sets
id_sets = {
    "entity": load_example_ids(entity_file),
    "short_phrase": load_example_ids(short_phrase_file),
    "date": load_example_ids(date_file),
    "number": load_example_ids(number_file),
    "number_with_unit": load_example_ids(number_with_unit_file),
}

# check pairwise intersections
print("Pairwise intersections:")
names = list(id_sets.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        overlap = id_sets[a] & id_sets[b]
        print(f"  {a} ∩ {b}: {len(overlap)}")

# check global intersection (ids present in ALL files)
all_common = set.intersection(*id_sets.values())
print(f"\nIn all files: {len(all_common)}")

# print examples per file
print("\nExamples per file:")
for name, ids in id_sets.items():
    print(f"  {name}: {len(ids)}")

In [ ]:
## Find the number of questions per type without the time sensitive ones
## Before running this cell, execute first 01_filter_mkqa_llm_judge.py to find the time-sensitive questions

DIR_IN = LOCAL_DATA_DIR + "/mkqa/time_filtering"

datasets = [
    "mkqa_answerable_entity_no_duplicates",
    "mkqa_answerable_short_phrase",
    "mkqa_answerable_date_no_duplicates",
    "mkqa_answerable_number_no_duplicates",
    "mkqa_answerable_number_with_unit_no_duplicates",
]

for ds in datasets:
    filepath = os.path.join(DIR_IN, f"{ds}.jsonl")
    total, non_sensitive = 0, 0
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            total += 1
            if data.get("time_sensitive") == "NO":
                non_sensitive += 1
    print(f"{ds}: {non_sensitive}/{total} non-time-sensitive")

## Global MMLU

### Inspect the English dataset

In [ ]:
# load the HF dataset in english
from datasets import load_dataset
global_mmlu = load_dataset("CohereLabs/Global-MMLU", 'en')
mmlu_df = global_mmlu["test"].to_pandas()
dims = mmlu_df.shape
print(dims)

In [ ]:
## Print the number of questions per subject
subjects_ser = mmlu_df['subject'] # each column is a series
counts = subjects_ser.value_counts()
print(counts)
print(f"\nNum subjects: {counts.shape[0]}")

### Inspect the score distribution across questions 
Required: execution of ./experiments/01_filter_global_mmlu_regex.py and ./experiments/01_filter_global_mmlu_llm_judge.py

In [ ]:
import os
import json
import math
import matplotlib.pyplot as plt
from collections import Counter
from matplotlib.patches import Patch

DATA_DIR = "../data/global_mmlu/en_gpt-4.1_out"

# Manually define colors
# Muted colors for 1–7, strong colors for 8–10
SCORE_COLORS = {
    1: "#d9d9d9",  # light gray
    2: "#cfcfcf",
    3: "#c4c4c4",
    4: "#b8b8b8",
    5: "#adadad",
    6: "#a1a1a1",
    7: "#969696",
    8: "#1f77b4",  # strong blue
    9: "#ff7f0e",  # strong orange
    10: "#2ca02c", # strong green
}

In [ ]:
# -------- CONFIG --------
OUTPUT_FILE = DATA_DIR + "score_distribution.pdf"

# -------- GLOBAL STYLE (publication-friendly) --------
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "legend.fontsize": 9,
})

# -------- LOAD DATA --------
scores_per_split = {}

for fname in sorted(os.listdir(DATA_DIR)):
    if not fname.endswith(".jsonl"):
        continue

    split = fname.replace(".jsonl", "")
    path = os.path.join(DATA_DIR, fname)

    scores = []

    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
                score = obj.get("llm_score", None)

                if isinstance(score, int) and 1 <= score <= 10:
                    scores.append(score)

            except json.JSONDecodeError:
                print(f"[WARN] Skipping bad JSON in {fname} at line {line_num}")

    if scores:
        scores_per_split[split] = scores

# -------- PLOT LAYOUT --------
n = len(scores_per_split)
cols = min(7, n)  # max 4 per row for readability
rows = math.ceil(n / cols)

plt.figure(figsize=(cols * 3.2, rows * 3.2))  # compact for paper

# -------- PLOT PIE CHARTS --------
for i, (split, scores) in enumerate(scores_per_split.items(), start=1):
    plt.subplot(rows, cols, i)

    counts = Counter(scores)

    labels = sorted(counts.keys())
    sizes = [counts[k] for k in labels]
    colors = [SCORE_COLORS[k] for k in labels]

    plt.pie(
        sizes,
        labels=labels,
        colors=colors,
        autopct=lambda p: f"{p:.0f}%" if p >= 5 else "",
        startangle=90,
        textprops={"fontsize": 9}
    )

    plt.title(split, fontsize=10)

# -------- GLOBAL LEGEND --------
legend_elements = [
    Patch(facecolor=SCORE_COLORS[s], label=f"{s}")
    for s in range(1, 11)
]

plt.legend(
    handles=legend_elements,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    title="Score",
    title_fontsize=10
)

# -------- FINALIZE & SAVE --------
plt.tight_layout()

# Save as vector PDF (best for papers)
plt.savefig(OUTPUT_FILE, bbox_inches="tight")

# Optional: also save PNG
plt.savefig(DATA_DIR + "score_distribution.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
all_scores = []

# Load all scores
for fname in sorted(os.listdir(DATA_DIR)):
    if not fname.endswith(".jsonl"):
        continue

    path = os.path.join(DATA_DIR, fname)
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                score = obj.get("llm_score", None)
                if isinstance(score, int) and 1 <= score <= 10:
                    all_scores.append(score)
            except json.JSONDecodeError:
                print(f"[WARN] Skipping bad JSON in {fname} at line {line_num}")

# Count scores
counts = Counter(all_scores)
total = sum(counts.values())

# Print counts + percentages
print("=== LLM Score Distribution (All Splits Combined) ===")
print(f"Total examples: {total}\n")

for score in range(1, 11):
    c = counts.get(score, 0)
    pct = (c / total * 100) if total > 0 else 0
    print(f"Score {score:>2}: {c:>6}  ({pct:5.2f}%)")

# Plot pie
labels = [s for s in range(1, 11) if counts.get(s, 0) > 0]
sizes = [counts[s] for s in labels]
colors = [SCORE_COLORS[s] for s in labels]

plt.figure(figsize=(8, 8))
plt.pie(
    sizes,
    labels=labels,
    colors=colors,
    autopct=lambda p: f"{p:.1f}%" if p >= 2 else "",
    startangle=90
)
plt.title("LLM Score Distribution (All Splits Combined)")

legend_elements = [
    Patch(facecolor=SCORE_COLORS[s], label=f"Score {s}")
    for s in range(1, 11)
]

plt.legend(
    handles=legend_elements,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    title="LLM Score"
)

plt.tight_layout()
plt.show()


In [ ]:
INPUT_DIR = "../data/global_mmlu/en_gpt-4.1_out/"
OUTPUT_DIR = "../data/global_mmlu/en_final/"
LLM_SCORE_THRESHOLD = 8

os.makedirs(OUTPUT_DIR, exist_ok=True)

total_before = 0   # total valid scored examples
total_after = 0    # total kept examples

for fname in sorted(os.listdir(INPUT_DIR)):
    if not fname.endswith(".jsonl"):
        continue

    input_path = os.path.join(INPUT_DIR, fname)
    output_path = os.path.join(OUTPUT_DIR, fname)

    kept = 0
    seen = 0  # count ONLY valid scored items

    valid_lines = []

    with open(input_path, "r", encoding="utf-8") as fin:
        for line_num, line in enumerate(fin, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
                score = obj.get("llm_score", None)

                # count only valid scores
                if isinstance(score, int) and 1 <= score <= 10:
                    seen += 1

                    if score >= LLM_SCORE_THRESHOLD:
                        valid_lines.append(obj)
                        kept += 1

            except json.JSONDecodeError:
                print(f"[WARN] Bad JSON in {fname} line {line_num}")

    # Warn if no valid entries
    if seen == 0:
        print(f"[WARN] {fname}: no valid scored examples")
        continue  # skip writing empty file

    # Write only if we kept something
    if kept > 0:
        with open(output_path, "w", encoding="utf-8") as fout:
            for obj in valid_lines:
                fout.write(json.dumps(obj) + "\n")
    else:
        print(f"[INFO] {fname}: no examples >= {LLM_SCORE_THRESHOLD}, skipping file")

    total_before += seen
    total_after += kept

    print(f"{fname}: kept {kept}/{seen} ({kept/seen:.2%})")

# -------- FINAL SUMMARY --------
print("\n=== Summary ===")

if total_before > 0:
    print(f"Total kept: {total_after}/{total_before} ({total_after/total_before:.2%})")
else:
    print("Total kept: 0/0 (no valid data)")

In [ ]:
import os
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -------- CONFIG --------
INITIAL_DIR = "../data/global_mmlu/en/"           # .csv files
FINAL_DIR   = "../data/global_mmlu/en_final/"     # .jsonl files
OUTPUT_FILE = FINAL_DIR + "retention_per_subject.pdf"

# -------- GLOBAL STYLE (publication-friendly) --------
plt.rcParams.update({
    "font.size":        10,
    "axes.titlesize":   10,
    "legend.fontsize":   9,

})

# -------- LOAD DATA --------
records = []

for fname in sorted(os.listdir(INITIAL_DIR)):
    if not fname.endswith(".csv"):
        continue
    stem       = fname[:-4]                         # e.g. "abstract_algebra"
    csv_path   = os.path.join(INITIAL_DIR, fname)
    jsonl_path = os.path.join(FINAL_DIR, stem + ".jsonl")

    # Total = rows in the initial CSV
    try:
        df    = pd.read_csv(csv_path)
        total = len(df)
    except Exception as e:
        print(f"[WARN] Could not read {fname}: {e}")
        continue

    # Kept = lines in the final JSONL (file may not exist if nothing passed)
    kept = 0
    if os.path.exists(jsonl_path):
        with open(jsonl_path, "r", encoding="utf-8") as f:
            kept = sum(1 for line in f if line.strip())

    label = stem.replace("_", " ").capitalize()
    records.append({"label": label, "kept": kept, "total": total})

if not records:
    raise RuntimeError("No data found. Check INITIAL_DIR and FINAL_DIR paths.")

# # Sort by retention rate descending; push zero-total entries to bottom
# records.sort(
#     key=lambda r: (r["total"] > 0, r["kept"] / r["total"] if r["total"] > 0 else 0),
#     reverse=True,
# )

# Sort by subject in alphabetical order
records.sort(key=lambda r: r["label"])

labels   = [r["label"] for r in records]
kept     = np.array([r["kept"]  for r in records])
total    = np.array([r["total"] for r in records])
filtered = total - kept
pct      = np.where(total > 0, kept / total * 100, np.nan)

# -------- STYLE --------
COLOR_KEPT     = "#378ADD"
COLOR_FILTERED = "#D3D1C7"
COLOR_SPINE    = "#CCCCC5"

# -------- PLOT --------
n          = len(labels)
BAR_HEIGHT = 0.62
fig, ax    = plt.subplots(figsize=(10, n * 0.32 + 1.6))
y          = np.arange(n)

ax.barh(y, kept,     height=BAR_HEIGHT, color=COLOR_KEPT,     label="Kept (gpt_4.1 score ≥ 8)")
ax.barh(y, filtered, height=BAR_HEIGHT, left=kept,
        color=COLOR_FILTERED, label="Filtered out")

# Retention-% labels at the right end of each full bar
for i in range(n):
    if not np.isnan(pct[i]):
        ax.text(total[i] + 4, i, f"{pct[i]:.0f}%",
                va="center", ha="left", fontsize=7.5, color="#5F5E5A")
    else:
        ax.text(6, i, "no valid data", va="center", ha="left",
                fontsize=7.5, color="#B4B2A9", style="italic")

# Axes
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8.5)
ax.set_xlabel("Number of questions", fontsize=9, color="#5F5E5A")
ax.tick_params(axis="x", labelsize=8.5, colors="#5F5E5A")
ax.tick_params(axis="y", colors="#5F5E5A")
ax.set_xlim(0, (total.max() if total.max() > 0 else 1) * 1.12)
ax.set_ylim(-0.6, n - 0.4)
ax.invert_yaxis()
ax.spines["bottom"].set_color(COLOR_SPINE)
ax.grid(axis="x", color="#E8E7E0", linewidth=0.6, zorder=0)
ax.set_axisbelow(True)

# Summary stats box
valid_mask  = total > 0
total_kept  = int(kept[valid_mask].sum())
total_eval  = int(total[valid_mask].sum())
overall_pct = total_kept / total_eval * 100 if total_eval > 0 else 0
no_data     = int((~valid_mask).sum())

# Legend
ax.legend(
    handles=[
        mpatches.Patch(color=COLOR_KEPT,     label="Kept (gpt_4.1 score ≥ 8)"),
        mpatches.Patch(color=COLOR_FILTERED, label="Filtered out"),
    ],
    fontsize=8.5, frameon=False,
    loc="lower right",
)

# -------- FINALIZE & SAVE --------
fig.tight_layout()

# Save as vector PDF (best for papers)
fig.savefig(OUTPUT_FILE, bbox_inches="tight")

# Optional: also save PNG
fig.savefig(OUTPUT_FILE.replace(".pdf", ".png"), dpi=300, bbox_inches="tight")
print(f"Saved: {OUTPUT_FILE}  +  .png")
plt.show()


In [ ]:
import os, json
from collections import Counter
import matplotlib.pyplot as plt

OUTPUT_DIR = "../data/global_mmlu/en_final/"

# check the types of questions in the dataset
# distringuish between mostly: math calculations, mixed, knowledge
math_calculations = ['abstract_algebra', 'college_computer_science', 'college_mathematics', 'elementary_mathematics', 
                     'formal_logic', 'high_school_mathematics', 'high_school_physics', 'high_school_statistics', 'professional_accounting']

mixed = ['college_chemistry', 'college_physics', 'conceptual_physics', 'electrical_engineering', 'high_school_chemistry', 'high_school_computer_science', 
         'high_school_macroeconomics', 'machine_learning']


counts = Counter()

for fname in os.listdir(OUTPUT_DIR):
    if not fname.endswith(".jsonl"):
        continue

    input_path = os.path.join(OUTPUT_DIR, fname)

    with open(input_path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)

            subject = obj.get("subject")  # adjust key if needed

            if subject in math_calculations:
                counts["math_calculations"] += 1
            elif subject in mixed:
                counts["mixed"] += 1
            else:
                counts["knowledge"] += 1



labels = list(counts.keys())
sizes = list(counts.values())

plt.figure()
plt.pie(sizes, labels=labels, autopct='%1.1f%%')
plt.title("(Rough) GMMLU Final Dataset Composition by Question Type")
plt.show()

### Generate the dataset for the rest of the languages

In [ ]:
# To be executed after running 01_filter_global_mmlu_regex.py and 01_filter_global_mmlu_llm_judge.py
from pathlib import Path
import pandas as pd
from datasets import load_dataset

LANGUAGES = ["fr", "es", "pl", "ru", "ja"]
EN_DIR = Path(LOCAL_DATA_DIR) / 'global_mmlu/en_final'

for lang in LANGUAGES:

    # Create the language's output dir if it does not exist
    output_dir = Path(LOCAL_DATA_DIR) / f'global_mmlu/{lang}_final'
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load the dataset in the language
    global_mmlu = load_dataset("CohereLabs/Global-MMLU", lang)
    mmlu_lang_df = global_mmlu["test"].to_pandas()

    # Iterate over the datasplits available in en_final
    for f in sorted(EN_DIR.glob("*.jsonl")):
        datasplit = f.stem  # e.g. "abstract_algebra", "anatomy", ...

        # Read the kept English questions for this datasplit
        en_df = pd.read_json(f, lines=True)

        # Read the datasplit in the considered language
        lang_df = mmlu_lang_df[mmlu_lang_df['subject'] == datasplit]

        # Keep only the "sample_ids" that are present in English
        valid_ids = set(en_df["sample_id"])
        lang_filtered_df = lang_df[lang_df["sample_id"].isin(valid_ids)]
        out_path = output_dir / f"{datasplit}.jsonl"
        lang_filtered_df.to_json(out_path, orient="records", lines=True, force_ascii=False)

In [ ]:
FR_DATA_DIR = Path(LOCAL_DATA_DIR) / 'global_mmlu/fr_final'
FR_RESULTS_DIR = Path(LOCAL_RESULTS_DIR) / 'global_mmlu/llama_3.1_8B'

# Count questions in fr_final (files named: {datasplit}.jsonl)
data_counts = pd.Series({
    f.stem: len(pd.read_json(f, lines=True))
    for f in sorted(FR_DATA_DIR.glob("*.jsonl"))
})

# Count questions in results dir (files named: {datasplit}/fr_answer.jsonl)
results_counts = pd.Series({
    f.parent.name: len(pd.read_json(f, lines=True))
    for f in sorted(FR_RESULTS_DIR.glob("*/fr_answer.jsonl"))
})

# Compare side by side
comparison = pd.DataFrame({
    "fr_final": data_counts,
    "llama_results": results_counts,
}).sort_index()

comparison["diff"] = comparison["llama_results"] - comparison["fr_final"]

print(comparison.to_string())
print(f"\nTotal fr_final:     {comparison['fr_final'].sum()}")
print(f"Total llama_results: {comparison['llama_results'].sum()}")
print(f"Total diff:          {comparison['diff'].sum()}")

## Train, eval, test (0.7: 0.15: 0.15) stratified split

In [ ]:
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
import random
from collections import Counter
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.constants import LOCAL_RESULTS_DIR, LOCAL_DATA_DIR, MODELS, DATASPLITS_MKQA, DATASPLITS_GMMLU

lang = 'ja'
SEED = 42
random.seed(SEED)

train_frac = 0.7
val_frac   = 0.15
test_frac  = 0.15

dataset = "mkqa"
datasplits = DATASPLITS_MKQA if dataset == 'mkqa' else DATASPLITS_GMMLU

In [ ]:
# Stratified splitting to preserve correctness ratios
# CONSIDER ONLY THE QUESTIONS THAT ARE NOT TIME SENSITIVE

TIME_FILTER_ROOT = f"{LOCAL_DATA_DIR}/{dataset}/time_filtering"

for ds in datasplits:
    print(f"\n\nStratified splitting for {ds} (time_sensitive == NO only)")

    # 1) Load allowed (non-time-sensitive) ids for this split
    allowed_ids = set()
    tf_path = Path(TIME_FILTER_ROOT) / f"{ds}.jsonl"
    if not tf_path.exists():
        raise FileNotFoundError(f"Missing time filtering file: {tf_path}")

    with tf_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            d = json.loads(line)
            if d.get("time_sensitive") == "NO":
                allowed_ids.add(int(d["example_id"]))

    print(f"Num of non-time sensitive ids for {ds} loaded: {len(allowed_ids)}")

    for llm in MODELS:
        print(f"\nStratified splitting for {llm}-{ds} (time_sensitive == NO only)")
        
        # 2) Load correctness data
        corr_path = Path(LOCAL_RESULTS_DIR) / f"{dataset}/{llm}/{ds}/{lang}_correctness.jsonl"
        with corr_path.open("r", encoding="utf-8") as fin:
            correctness_ds = json.loads(fin.readline()) 

        # 3) Filter correctness to time-insensitive ids only
        filtered_items = []
        missing_in_filter = 0

        for ex_id_str, label in correctness_ds.items():
            try:
                ex_id = int(ex_id_str)
            except ValueError:
                continue

            if ex_id in allowed_ids:
                filtered_items.append((ex_id_str, label))
            else:
                missing_in_filter += 1

        if not filtered_items:
            print("WARNING: After filtering, there are 0 examples. Skipping.")
            continue

        all_ids = [x[0] for x in filtered_items]
        all_labels = [x[1] for x in filtered_items]

        print(f"Correctness total: {len(correctness_ds)}")
        print(f"Kept after NO-filter: {len(all_ids)} (dropped {missing_in_filter})")
        print("Label dist after filter:", Counter(all_labels))

        # 4) Split (stratified)
        train_ids, temp_ids, train_lbls, temp_lbls = train_test_split(
            all_ids,
            all_labels,
            test_size=1 - train_frac,
            stratify=all_labels,
            random_state=SEED,
        )

        relative_test_size = test_frac / (val_frac + test_frac)
        val_ids, test_ids, val_lbls, test_lbls = train_test_split(
            temp_ids,
            temp_lbls,
            test_size=relative_test_size,
            stratify=temp_lbls,
            random_state=SEED,
        )

        # verify class balance
        print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")
        print("Train label dist:", Counter(train_lbls))
        print("Val label dist:", Counter(val_lbls))
        print("Test label dist:", Counter(test_lbls))

        # 5) Save
        splits = {
            "train": list(train_ids),
            "val": list(val_ids),
            "test": list(test_ids),
        }
        out_path = Path(LOCAL_RESULTS_DIR) / (
            f"{dataset}/{llm}/{ds}/train_lang_{lang}_splits_stratified_without_time_sensitive.json"
        )
        if not out_path.exists():
            with out_path.open("w", encoding="utf-8") as f:
                json.dump(splits, f, ensure_ascii=False, indent=2)
            print(f"Saved splits to: {out_path}")
        else:
            print(f"File {out_path} already exists")

In [ ]:
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
import random
from collections import Counter
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from src.constants import LOCAL_RESULTS_DIR, LOCAL_DATA_DIR, MODELS, DATASPLITS_MKQA, DATASPLITS_GMMLU

lang = 'ru'
SEED = 42
random.seed(SEED)

train_frac = 0.7
val_frac   = 0.15
test_frac  = 0.15

dataset = "global_mmlu"
datasplits = DATASPLITS_MKQA if dataset == 'mkqa' else DATASPLITS_GMMLU

In [ ]:
# Stratified splitting to preserve correctness ratios
for ds in datasplits:
    print(f"\n\nStratified splitting for {ds}")
    for llm in MODELS:
        print(f"\nStratified splitting for {llm}-{ds}")

        # 2) Load correctness data
        corr_path = Path(LOCAL_RESULTS_DIR) / f"{dataset}/{llm}/{ds}/{lang}_correctness.jsonl"
        if not corr_path.exists():
            print(f"  [SKIP] File not found: {corr_path}")
            continue
        with corr_path.open("r", encoding="utf-8") as fin:
            line = fin.readline()
        if not line.strip():
            print(f"  [SKIP] Empty file: {corr_path}")
            continue
        correctness_ds = json.loads(line)
        if not correctness_ds:
            print(f"  [SKIP] Empty correctness dict: {corr_path}")
            continue

        # 3) Get items
        all_items = list(correctness_ds.items())
        all_ids = [x[0] for x in all_items]
        all_labels = [x[1] for x in all_items]
        print(f"Correctness total: {len(correctness_ds)}")
        print(f"Valid items: {len(all_ids)}")
        print(f"Label dist: {Counter(all_labels)}")
        if len(set(all_labels)) < 2:
            print(f"[SKIP] Only one class present ({set(all_labels)}), can't stratify")
            continue

        # 4) First split (stratified with fallback)
        try:
            train_ids, temp_ids, train_lbls, temp_lbls = train_test_split(
                all_ids,
                all_labels,
                test_size=1 - train_frac,
                stratify=all_labels,
                random_state=SEED,
            )
        except ValueError:
            print("  [WARN] Too few minority samples to stratify train/temp — falling back to random split")
            train_ids, temp_ids, train_lbls, temp_lbls = train_test_split(
                all_ids,
                all_labels,
                test_size=1 - train_frac,
                random_state=SEED,
            )

        # Second split (stratified with fallback, handle tiny temp sets)
        relative_test_size = test_frac / (val_frac + test_frac)
        if len(temp_ids) < 2:
            print(f"  [WARN] Only {len(temp_ids)} sample(s) in temp — assigning all to val, test will be empty")
            val_ids, val_lbls = list(temp_ids), list(temp_lbls)
            test_ids, test_lbls = [], []
        else:
            try:
                val_ids, test_ids, val_lbls, test_lbls = train_test_split(
                    temp_ids,
                    temp_lbls,
                    test_size=relative_test_size,
                    stratify=temp_lbls,
                    random_state=SEED,
                )
            except ValueError:
                print("  [WARN] Too few minority samples to stratify val/test — falling back to random split")
                val_ids, test_ids, val_lbls, test_lbls = train_test_split(
                    temp_ids,
                    temp_lbls,
                    test_size=relative_test_size,
                    random_state=SEED,
                )

        # Verify class balance
        print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")
        print(f"Train label dist: {Counter(train_lbls)}")
        print(f"Val label dist:   {Counter(val_lbls)}")
        print(f"Test label dist:  {Counter(test_lbls)}")

        # 5) Save
        splits = {
            "train": list(train_ids),
            "val": list(val_ids),
            "test": list(test_ids),
        }
        out_path = Path(LOCAL_RESULTS_DIR) / f"{dataset}/{llm}/{ds}/train_lang_{lang}_splits_stratified.json"
        if out_path.exists():
            print(f"  [SKIP] File already exists: {out_path}")
            continue
        with out_path.open("w", encoding="utf-8") as f:
            json.dump(splits, f, ensure_ascii=False, indent=2)
        print(f"Saved splits to: {out_path}")